In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tantiya sa paggamit: isang minuto sa Eagle processor (PAALALA: Ito ay tantiya lamang. Maaaring mag-iba ang iyong runtime.)*

## Background

Ang circuit-knitting ay isang pangkalahatang termino na sumasaklaw sa iba't ibang pamamaraan ng paghahati ng circuit sa maraming mas maliit na subcircuits na may mas kaunting gates at/o qubits. Ang bawat isa sa mga subcircuits ay maaaring isagawa nang independiyente at ang panghuling resulta ay nakukuha sa pamamagitan ng ilang classical post-processing sa kinalabasan ng bawat subcircuit. Ang pamamaraang ito ay maaaring ma-access sa [circuit cutting Qiskit addon](https://qiskit.github.io/qiskit-addon-cutting/index.html), ang detalyadong paliwanag ng pamamaraan ay ibinigay sa [docs](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) kasama ng iba pang [introductory material](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Ang notebook na ito ay tumatalakay sa isang pamamaraang tinatawag na <b>wire cutting</b> kung saan ang circuit ay pinaghahati sa kahabaan ng wire [\[1\], \[2\]](#references). Tandaan na ang paghahati ay simple sa mga classical circuits dahil ang kinalabasan sa punto ng partition ay maaaring matukoy nang deterministiko, at ito ay 0 o 1 lamang. Gayunpaman, ang estado ng qubit sa punto ng pagputol ay, sa pangkalahatan, isang mixed state. Samakatuwid, ang bawat subcircuit ay kailangang sukatin nang maraming beses sa iba't ibang basis (karaniwang isang tomographically complete set ng basis tulad ng Pauli basis [\[3\], \[4\]](#references) at naaangkop na inihanda sa kanyang eigenstate. Ang Figure sa ibaba (<i>pasasalamat: PhD Thesis, Ritajit Majumdar</i>) ay nagpapakita ng halimbawa ng wire cutting para sa 4-qubit GHZ state sa tatlong subcircuits. Dito ang $M_j$ ay tumutukoy sa isang set ng basis (karaniwang Pauli X, Y at Z) at ang $P_i$ ay tumutukoy sa isang set ng eigenstates (karaniwang $|0\rangle$, $|1\rangle$, $|+\rangle$ at $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Dahil ang bawat subcircuit ay may mas kaunting qubits at/o gates, inaasahang mas hindi sila madaling maapektuhan ng noise. Ang notebook na ito ay nagpapakita ng halimbawa kung saan ang pamamaraang ito ay maaaring gamitin upang epektibong pigilin ang noise sa sistema.

## Mga Kinakailangan

Bago simulan ang tutorial na ito, tiyaking mayroon kayong mga sumusunod na naka-install:

- Qiskit SDK v2.0 o mas bago, na may [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization) support
- Qiskit Runtime v0.22 o mas bago ( `pip install qiskit-ibm-runtime` )
- Circuit cutting Qiskit addon v0.9.0 o mas bago (`pip install qiskit-addon-cutting`)

Isasaalang-alang natin ang isang Many Body Localization (MBL) circuit para sa notebook na ito. Ang MBL circuit ay isang hardware-efficient circuit at may parameter na dalawang parameters na $\theta$ at $\vec{\phi}$. Kapag ang $\theta$ ay nakatakda sa $0$ at ang initial state ay inihanda sa $|0\rangle$ para sa lahat ng qubits, ang ideal expectation value ng $\langle Z_i \rangle$ ay $+1$ para sa bawat qubit site $i$ anuman ang mga halaga ng $\vec{\phi}$. Maaari ninyong tingnan ang mas maraming detalye tungkol sa MBL circuits sa <a href="https://arxiv.org/abs/2307.07552">papel na ito</a>.

## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Bahagi I. Halimbawa ng maliit na sukat

### Hakbang 1: I-map ang classical inputs sa quantum problem

Una, bubuo tayo ng template circuit na walang partikular na parameter values. Nagbibigay din tayo ng mga placeholders, na tinatawag na `CutWire`, upang markahan ang posisyon ng mga pagputol. Para sa halimbawa ng maliit na sukat, isasaalang-alang natin ang 10-qubit MBL circuit.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Ngayon ay markahan natin ang circuit para sa pagputol sa pamamagitan ng paglalagay ng wastong **CutWire** upang lumikha ng dalawang halos pantay na pagputol. Itinakda natin ang `use_cut=True` sa function, at papayagan itong markahan pagkatapos ng $\frac{n}{2}$ qubits, kung saan ang $n$ ay ang bilang ng mga qubits sa orihinal na circuit.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Hakbang 2: I-optimize ang problema para sa quantum hardware execution
Susunod, puputulin natin ang circuit sa dalawang mas maliit na subcircuits. Para sa halimbawang ito, manatili tayo sa 2 subcircuits lamang. Para dito, gagamitin natin ang <a href="https://qiskit.github.io/qiskit-addon-cutting/">Qiskit Addon: Circuit Cutting</a>.
#### Putulin ang circuit sa mas maliit na subcircuits
Ang pagputol ng wire sa isang punto ay nagdaragdag ng isa sa bilang ng qubit. Bukod sa orihinal na qubit, mayroon na ngayong karagdagang qubit bilang placeholder sa circuit pagkatapos ng pagputol. Ang sumusunod na larawan ay nagbibigay ng representasyon:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Ang Addon na ito ay gumagamit ng function na `cut_wires` upang isaalang-alang ang mga karagdagang qubits na lumilitaw dahil sa pagputol.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Lumikha at palawakin ang mga observables
Ngayon ay bubuo tayo ng observable na $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Dahil ang ideal na kinalabasan ng $\langle Z_i \rangle$ para sa bawat $i$ ay $+1$, ang ideal na kinalabasan ng $M_z$ ay $+1$ din.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Gayunpaman, pansinin na ang bilang ng mga qubits sa circuit ay tumaas pagkatapos ng paglalagay ng virtual 2-qubit `Move` operations pagkatapos ng pagputol. Samakatuwid, kailangan din nating palawakin ang mga observables sa pamamagitan ng paglalagay ng mga identities upang mag-assert sa kasalukuyang circuit.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Tingnan natin ang mga subcircuits

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Ang mga observables ay pinaghati rin upang umangkop sa mga subcircuits

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

Pansinin na ang bawat subcircuit ay humahantong sa ilang samples. Isinasaalang-alang ng reconstruction ang kinalabasan ng bawat isa sa mga samples na ito. Ang bawat isa sa mga samples na ito ay tinatawag na `subexperiment`.
Ang pagpapalawak ng observable gamit ang `Move` operation ay nangangailangan ng `PauliList` data structure. Maaari din tayong lumikha ng $M_z$ observable sa mas generic na `SparsePauliOp` data structure na magiging kapaki-pakinabang mamaya sa panahon ng reconstruction ng mga subexperiments.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Tingnan natin ang dalawang halimbawa kung saan ang mga cut qubits ay sinusukat sa dalawang magkaibang basis. Una, ito ay sinusukat sa normal na Z basis, at susunod ay sinusukat sa X basis.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### I-transpile ang bawat subexperiment

Sa kasalukuyan ay kailangan nating i-transpile ang ating mga circuits bago isumite ang mga ito para sa execution. Samakatuwid, ita-transpile natin muna ang bawat circuit sa mga subexperiments.